# Imports

In [2]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
from IPython.display import display, Markdown
import json
from pathlib import Path
from datetime import datetime as dt

# EDA

In [3]:
# read data and rename columns
df = pd.read_csv("data/requirements.csv")
df.columns = ["req", "s1", "s2", "s3", "s4", "s5", "s6"]

In [4]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Find Examples

In [5]:
# collect used examples to exclude from test dataset
indexes_to_drop = []

In [6]:
# collect examples with single fault
examples = {}

for c in df.columns[1:]:
    examples[c] = {}
    i = df.loc[ (df[c] == 1) & (df[df.columns[1:][df.columns[1:]!=c]].sum(axis=1) == 0) ].sample(1)
    indexes_to_drop.append(i.index)
    i = i.values[0]
    examples[c]["req"] = i[0]
    examples[c]["vec"] = "[" + ",".join(map(str, i[1:])) + "]"

In [7]:
# collect examples with multiple faults
examples_multiple = {}

t = df.loc[df[df.columns[1:]].sum(axis=1) == 2]
idx = t.drop(columns="req").drop_duplicates().index
indexes_to_drop.append(idx)
t = df.iloc[idx, :].values

for r in t:
    c = "&".join(df.columns[1:][r[1:]==1])
    examples_multiple[c] = {}
    examples_multiple[c]["req"] = r[0]
    examples_multiple[c]["vec"] = "[" + ",".join(map(str, r[1:])) + "]"

In [8]:
# add both single and multiple into one place
examples.update(examples_multiple)

In [9]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e in examples.values():
    examples_txt += f"Requirement: {e['req']}\n"
    examples_txt += f"Vector: {e['vec']}\n"
    examples_txt += "\n"

In [10]:
print("\n".join(examples_txt.split("\n")[:5]))

Requirement: The pedal assembly must include a mechanical release mechanism that ensures the return of the pedal to the idle position when not actively engaged by the driver
Vector: [1,0,0,0,0,0]

Requirement: In critical situations, the vehicle must enable an emergency steering assist feature to aid the driver in maintaining control of the vehicle
Vector: [0,1,0,0,0,0]


In [11]:
# drop indexes used in examples
indexes_to_drop = np.concatenate(indexes_to_drop)
df.drop(index=indexes_to_drop, inplace=True)
df.reset_index(drop=True, inplace=True)

# LLM

In [12]:
from prompts.SystemPrompts import SystemPrompt
from prompts.Sensors import Sensors
from prompts.UserPrompt import UserPrompt

In [13]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

In [14]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.5
)

In [15]:
# system prompt
messages = [
    {'role': 'system',
     'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt)}
]

In [16]:
# select random instance
instance = df.sample(1)
idx = instance.index
req = instance.values[0][0]
vec = "[" + ",".join(map(str, instance.values[0][1:])) + "]"

In [17]:
# add user prompt
messages.append({"role":"user", "content":UserPrompt.format(req=req)})

In [18]:
# run LLM
response = llm.invoke(messages)
messages.append({"role":"assistant", "content":response.content})

In [19]:
def parse_result(messages):
    '''Parse the LLM result'''
    return messages[-1]["content"].replace(" ", "")

In [20]:
# is the result match the original
parse_result(messages) == vec

True

In [21]:
# number of tokens
response.response_metadata["token_usage"]

{'completion_tokens': 18, 'prompt_tokens': 782, 'total_tokens': 800}

# Save conversation

In [22]:
# save conversation and metadata
time = dt.now()

conv_file = Path().cwd() / f"chats/conv_{time.strftime('%m.%d.%Y-%H:%M:%S')}.json"
conv_file.parent.mkdir(exist_ok=True)
conv_file.touch()

with conv_file.open("w") as f:
    json.dump({'messages': messages,
               'true_solution': vec,
               'is_correct': parse_result(messages) == vec,
               'token_usage': response.response_metadata["token_usage"]},
            f, indent=4)